In [ ]:
%run EEG_dataloader.ipynb

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

path = Path('data')
pre_processed_path = path / 'pre_processed'

RESAMPLING_RATE = 64


train_segments = torch.load(
    path / 'segments_h1k_train.pt',
    weights_only=True
)

val_segments = torch.load(
    path / 'segments_h1k_val.pt',
    weights_only=True
)

test_segments = torch.load(
    path / 'segments_h1k_test.pt',
    weights_only=True
)

print("Training patients:", len(train_segments))
print("Validation patients:", len(val_segments))
print("Test patients:", len(test_segments))

Training patients: 825
Validation patients: 103
Test patients: 104


In [ ]:

import torch
import torch.nn as nn

class EEG_CNN(nn.Module):
    def __init__(self, n_channels=20):
        super().__init__()

        self.n_channels = n_channels

        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=15, padding=7),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(4),

            nn.Conv1d(16, 32, kernel_size=9, padding=4),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(4),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(4),

            nn.AdaptiveAvgPool1d(4)
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_channels * 64 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        B, C, T = x.shape

        assert C == self.n_channels

        x = x.reshape(B * C, 1, T)

        x = self.features(x)


        x = x.reshape(B, C * 64 * 4)

        x = self.regressor(x)

        return x.squeeze(1)


In [ ]:

import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader

SEGMENT_DURATION = 16
RESAMPLING_RATE = 64
BATCH_SIZE = 16

def create_windows(segments, n_windows=12):
    window_segments = []

    for segment in segments:
        duration = min(float(segment[1]), 900.0)
        age = segment[3]
        filename = segment[4]

        starts = np.linspace(
            0,
            duration - SEGMENT_DURATION,
            n_windows
        )

        for start in starts:
            window_segments.append((
                0,
                duration,
                float(start),
                age,
                filename
            ))

    return window_segments

train_windows = create_windows(train_segments, n_windows=12)
val_windows = create_windows(val_segments, n_windows=20)

print("Training windows:", len(train_windows))
print("Validation windows:", len(val_windows))


Training windows: 9900
Validation windows: 2060


In [ ]:

train_dataset = EEGDataset(
    file_path=path,
    segments=train_windows,
    segment_duration=SEGMENT_DURATION,
    sampling_rate=RESAMPLING_RATE,
    cache_size=None
)

val_dataset = EEGDataset(
    file_path=path,
    segments=val_windows,
    segment_duration=SEGMENT_DURATION,
    sampling_rate=RESAMPLING_RATE,
    cache_size=None
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

x, y, idx = next(iter(train_loader))

print("EEG input:", x.shape)
print("Labels:", len(y))


EEG input: torch.Size([16, 20, 1024])
Labels: 16


In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
import copy

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = EEG_CNN(n_channels=20).to(device)

criterion = nn.MSELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

def train_one_epoch():
    model.train()

    total_loss = 0.0

    for x, y, _ in train_loader:
        x = (x / 100.0).to(device)
        y = torch.as_tensor(
            y, dtype=torch.float32, device=device
        )

        optimizer.zero_grad()

        predictions = model(x)
        loss = criterion(predictions, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(y)

    return total_loss / len(train_dataset)


In [ ]:

from collections import defaultdict
from sklearn.metrics import mean_absolute_error

@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    predictions_by_patient = defaultdict(list)
    ages_by_patient = {}

    for x, y, indices in loader:
        x = (x / 100.0).to(device)

        predictions = model(x).cpu().numpy()

        for pred, age, idx in zip(
            predictions, y, indices
        ):
            idx = int(idx)

            patient = loader.dataset.segments[idx][4]

            predictions_by_patient[patient].append(
                float(pred)
            )
            ages_by_patient[patient] = float(age)

    true_ages = []
    predicted_ages = []

    for patient, preds in predictions_by_patient.items():
        true_ages.append(ages_by_patient[patient])
        predicted_ages.append(np.mean(preds))

    mae = mean_absolute_error(
        true_ages, predicted_ages
    )

    return mae, np.array(true_ages), np.array(predicted_ages)


In [ ]:
#RUN 1
EPOCHS = 30
PATIENCE = 7

best_mae = float("inf")
best_weights = None
epochs_without_improvement = 0

train_losses = []
val_maes = []

for epoch in range(EPOCHS):
    train_loss = train_one_epoch()
    val_mae, _, _ = evaluate(model, val_loader)

    train_losses.append(train_loss)
    val_maes.append(val_mae)

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train MSE: {train_loss:.3f} | "
        f"Validation MAE: {val_mae:.3f}"
    )

    if val_mae < best_mae:
        best_mae = val_mae
        best_weights = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping")
        break

model.load_state_dict(best_weights)

print(f"Best validation MAE: {best_mae:.3f} years")


KeyboardInterrupt: 

In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(train_losses)
plt.xlabel("Epoch")
plt.ylabel("Training MSE")
plt.title("CNN training loss")
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(val_maes, label="CNN validation MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE (years)")
plt.title("CNN vs baseline")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
model.load_state_dict(best_weights)

In [ ]:

#60 second windows with 30 second overlap

OVERLAP_DURATION = 60
STRIDE = 30

def create_overlapping_windows(segments):
    windows = []

    for segment in segments:
        duration = min(float(segment[1]), 900.0)
        age = segment[3]
        filename = segment[4]

        for start in range(
            0,
            int(duration) - OVERLAP_DURATION + 1,
            STRIDE
        ):
            windows.append((
                0, duration, start, age, filename
            ))

    return windows

train_overlap = create_overlapping_windows(train_segments)
val_overlap = create_overlapping_windows(val_segments)

print("Train windows:", len(train_overlap))
print("Validation windows:", len(val_overlap))


In [ ]:

train_overlap_dataset = EEGDataset(
    file_path=path,
    segments=train_overlap,
    segment_duration=OVERLAP_DURATION,
    sampling_rate=RESAMPLING_RATE,
    cache_size=None
)

val_overlap_dataset = EEGDataset(
    file_path=path,
    segments=val_overlap,
    segment_duration=OVERLAP_DURATION,
    sampling_rate=RESAMPLING_RATE,
    cache_size=None
)

train_overlap_loader = DataLoader(
    train_overlap_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

val_overlap_loader = DataLoader(
    val_overlap_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

x, y, _ = next(iter(train_overlap_loader))

print("Input:", x.shape)



In [ ]:

import copy
import torch.optim as optim

overlap_model = EEG_CNN(n_channels=20).to(device)

overlap_optimizer = optim.Adam(
    overlap_model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

overlap_criterion = nn.MSELoss()

overlap_best_mae = float("inf")
overlap_best_weights = None
overlap_best_epoch = None
overlap_wait = 0

overlap_train_losses = []
overlap_val_maes = []

for epoch in range(30):
    overlap_model.train()
    total_loss = 0.0

    for x, y, _ in train_overlap_loader:
        x = (x / 100.0).to(device)
        y = torch.as_tensor(
            y, dtype=torch.float32, device=device
        )

        overlap_optimizer.zero_grad()

        predictions = overlap_model(x)
        loss = overlap_criterion(predictions, y)

        loss.backward()
        overlap_optimizer.step()

        total_loss += loss.item() * len(y)

    train_loss = total_loss / len(train_overlap_dataset)

    val_mae, _, _ = evaluate(
        overlap_model, val_overlap_loader
    )

    overlap_train_losses.append(train_loss)
    overlap_val_maes.append(val_mae)

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train MSE: {train_loss:.3f} | "
        f"Validation MAE: {val_mae:.3f}",
        flush=True
    )

    if val_mae < overlap_best_mae:
        overlap_best_mae = val_mae
        overlap_best_epoch = epoch + 1
        overlap_best_weights = copy.deepcopy(
            overlap_model.state_dict()
        )
        overlap_wait = 0
    else:
        overlap_wait += 1

    if overlap_wait >= 7:
        print("Early stopping")
        break

overlap_model.load_state_dict(overlap_best_weights)

print("Best epoch:", overlap_best_epoch)
print(f"Best validation MAE: {overlap_best_mae:.3f}")


varying results, best MAE ~0.562 - needs stabilizing